# 05. 시간 구간 제외하고 강종별 라벨 근거 정리

[03 LOT별 비교](03_lot_comparison.ipynb), [04 강종별 비교](04_grade_comparison.ipynb)와 [02 정비·부품교체 검토](02_maintenance_parts.ipynb)를 받아서 구간과 근거를 정리하는 곳임.
사용자 지정 시간 4개를 제외한 뒤 원본 22개 계측을 강종별로 비교할거임.
같은 강종의 다른 LOT, TC·OP 관계, 운전단계를 같이 보되 관측 통계를 정상 허용치로 바꾸지 않음.
현재 확정된 정상·이상 구간이 없어서 LABEL은 전부 NA로 남기고, 검토 근거는 모델 입력 X와 분리함.

In [ ]:
from pathlib import Path
import sys

import numpy as np
import pandas as pd
from IPython.display import display

ROOT = next(
    (path for path in (Path.cwd(), *Path.cwd().parents)
     if (path / "data").is_dir()
     and (path / "README.md").is_file()
     and (path / "notebooks").is_dir()),
    None,
)
if ROOT is None:
    raise FileNotFoundError("저장소 안에서 실행해야 함. data, README.md, notebooks 위치를 확인할 것")
if str(ROOT / "src") not in sys.path:
    sys.path.insert(0, str(ROOT / "src"))

from namelesspark_pipeline import (
    load_sources, build_lot_context, build_maintenance_context, apply_user_exclusions,
)

PROCESSED = ROOT / "data" / "processed"
OUT = PROCESSED / "review" / "v1"
GRADE_OUT = PROCESSED / "analysis" / "grade" / "v1"
SIGNAL_OUT = PROCESSED / "analysis" / "signals" / "v1"
for directory in [OUT, GRADE_OUT, SIGNAL_OUT]:
    directory.mkdir(parents=True, exist_ok=True)
pd.set_option("display.max_columns", 40)
pd.set_option("display.max_colwidth", 100)

## 1. LOT·강종과 정비 근거 불러오기

정비는 CAL01의 BM·TBM·CBM 전부 사용함. WO_NO로 부품교체이력을 연결하되 정비 1건당 1행을 유지한 표를 받음. 교체 없는 정비도 남아 있음. 한 작업에서 두 품목을 교체한 경우는 부품 상세 표에서 따로 확인함.

부품교체이력에는 실제 교체 시각이 없음. 아래 감지·착수·종료 시각은 정비이력의 시각이며 실제 부품교체 시각으로 보지 않음.

In [ ]:
sources = load_sources(ROOT)
temp = sources["temp"]
lot_context = build_lot_context(temp, sources["event"])
maintenance, part_details = build_maintenance_context(
    sources["maint"], sources["partchg"], sources["part"]
)
assert maintenance["WO_NO"].is_unique
assert len(maintenance) == sources["maint"]["EQP_TAG"].eq("P1-CR1-CAL01").sum()

maintenance_counts = maintenance.groupby("MNT_TYPE").agg(
    정비건수=("WO_NO", "size"), 교체있음=("HAS_PART_RECORD", "sum"),
    부품행수=("PART_RECORD_COUNT", "sum"),
)
maintenance_counts["교체없음"] = maintenance_counts["정비건수"] - maintenance_counts["교체있음"]
display(maintenance_counts.reindex(["BM", "TBM", "CBM"]))
display(lot_context)

## 2. 사용자가 정한 시간만 제외

| LOT | 제외할 시각 | 남길 경계 |
| --- | --- | --- |
| 240058 | 2024-06-09 08:00:00 이상 | 08:00 이전만 남김 |
| 240061 | 2024-06-15 02:30:00 미만 | 02:30부터 남김 |
| 240114 | 2024-11-04 20:00:00 이상 | 20:00 이전만 남김 |
| 240117 | 2024-11-13 09:00:00 미만 | 09:00부터 남김 |

원본은 그대로 두고 EXCLUDED로 표시함. 분 평균을 만들거나 새 시각을 채우지 않음. 예전 실험의 분당 관측 수 30개 조건, 240038 출력 전체 제외, 9·2·3 LOT 분할은 이번 기준에 적용하지 않음.

In [ ]:
row_review, exclusion_summary = apply_user_exclusions(temp)
pd.testing.assert_frame_equal(row_review[temp.columns], temp)
row_review["GRADE"] = row_review["LOT_NO"].map(lot_context.set_index("LOT_NO")["GRADE"])
assert row_review["GRADE"].notna().all()
assert not row_review.duplicated(["LOT_NO", "MEAS_DT"]).any()

row_counts = pd.DataFrame({
    "구분": ["원본", "사용자 지정 제외", "제외 후 남는 행"],
    "행수": [len(row_review), int(row_review["EXCLUDED"].sum()),
             int((~row_review["EXCLUDED"]).sum())],
})
assert row_counts.iloc[0]["행수"] == row_counts.iloc[1:]["행수"].sum()
display(exclusion_summary)
display(row_counts)

## 3. 강종별로 전체 계측 기준부터 검토

사용자 시간 제외 후 22개 계측을 7개 강종별로 요약하며, CP-Z4M은 보조계측으로 남겨 정의·단위·생성 규칙을 확인할거임.
[온도 데이터사전](https://app.notion.com/p/b5de609a45d083f89c01014faf0e4dda)의 강종군 5분류와 이동 폭은 재구성 가정이고, 건조 TC-Z1/Z2·OP-Z1/Z2와 CLN-IN은 강종 반응이 없다고 명시돼 있어 모든 센서를 일률적으로 보정하지 않음.
중앙값·5/95백분위·최솟값·최댓값은 정상 허용치가 아닌 관측 요약이며, 긴 LOT의 비중을 확인하려고 LOT별 중앙값 차이도 같이 남김.
SGCC·DP590·SPCD는 각각 1개 LOT라 강종 효과와 LOT 상태를 분리할 수 없고, 다른 강종도 설정값·허용편차·제어모드 확인 전까지 정상·이상 기준은 미확정임.
SPHC 240091의 TC-Z7/Z8 지속 수준 차이는 [04 강종별 비교](04_grade_comparison.ipynb)의 근거를 참조하고, 여기서는 판정 검토 표시만 남김.

In [ ]:
SENSOR_COLUMNS = [column for column in temp.columns if column not in ["LOT_NO", "MEAS_DT"]]
assert len(SENSOR_COLUMNS) == 22
assert all(pd.api.types.is_numeric_dtype(temp[column]) for column in SENSOR_COLUMNS)
retained = row_review.loc[~row_review["EXCLUDED"]].copy()
GRADE_INDEPENDENT_IN_DICTIONARY = {"CLN-IN", "TC-Z1", "TC-Z2", "OP-Z1", "OP-Z2"}
GRADE_RELEVANCE_SOURCE = "https://app.notion.com/p/b5de609a45d083f89c01014faf0e4dda"


def grade_relevance(sensor):
    if sensor in GRADE_INDEPENDENT_IN_DICTIONARY:
        return "사전상 강종 반응 없음; 관측 확인"
    if sensor == "CP-Z4M":
        return "강종 영향 검토; 보조계측 정의 확인 우선"
    return "강종 영향 검토; 사전의 수준 이동은 재구성 가정"


def sensor_review_context(sensor):
    if sensor == "CP-Z4M":
        return "보조계측", "", "정의·단위·생성 규칙, 다른 계측과의 관계"
    if sensor == "CP-Z4":
        return "분위기 계측", "", "계측 정의·단위, 강종별 목표 CP·허용폭·지속시간, 가스 운전·퍼지 기록"
    if sensor == "CLN-IN":
        return "세정 온도", "", "강종별 세정 목표온도·허용폭, 약품·속도·초기 운전상태"
    if sensor in ["TC-Z9", "TC-OUT1", "TC-OUT2"]:
        return "주기 변화가 관측된 온도", "", "강종별 목표·허용폭, 정상 주기·위상·진폭, 속도·유량·운전단계"
    if sensor.startswith("OP-"):
        return "제어출력", sensor.replace("OP-", "TC-"), "같은 존 TC 설정값, 자동·수동 모드, 출력 지속 허용시간, 실제 히터·밸브 동작"
    return "존 온도", sensor.replace("TC-", "OP-"), "강종·존별 설정온도, 허용편차·지속시간, 제어모드·라인속도·운전단계"


def observed_summary(series):
    valid = series.dropna()
    return {
        "VALID_ROWS": int(series.notna().sum()),
        "MISSING_ROWS": int(series.isna().sum()),
        "MEDIAN": valid.median(), "Q05": valid.quantile(0.05), "Q95": valid.quantile(0.95),
        "MIN": valid.min(), "MAX": valid.max(),
    }


lot_sensor_rows = []
for lot, group in retained.groupby("LOT_NO", sort=True):
    for sensor in SENSOR_COLUMNS:
        role, paired, needed = sensor_review_context(sensor)
        lot_sensor_rows.append({
            "LOT_NO": lot, "GRADE": group["GRADE"].iloc[0], "SENSOR": sensor,
            "RETAINED_ROWS": len(group), **observed_summary(group[sensor]),
            "SENSOR_ROLE": role, "PAIRED_SENSOR": paired,
            "GRADE_RELEVANCE": grade_relevance(sensor), "GRADE_RELEVANCE_SOURCE": GRADE_RELEVANCE_SOURCE,
            "SUMMARY_ROLE": "관측 요약; 정상 허용치 아님",
        })
lot_sensor_summary = pd.DataFrame(lot_sensor_rows)
assert len(lot_sensor_summary) == retained["LOT_NO"].nunique() * len(SENSOR_COLUMNS)
assert not lot_sensor_summary.duplicated(["LOT_NO", "SENSOR"]).any()
assert lot_sensor_summary["VALID_ROWS"].add(lot_sensor_summary["MISSING_ROWS"]).eq(lot_sensor_summary["RETAINED_ROWS"]).all()

# 지속 수준 차이의 통계·구간 비교는 04에서 관리하고 여기서는 근거만 참조함.
sphc_level_notes = {
    sensor: "04_grade_comparison.ipynb의 SPHC 240091 지속 수준 차이 검토 참조; 원인·정상 허용치 미확정"
    for sensor in ["TC-Z7", "TC-Z8"]
}
lot_sensor_summary["SUSTAINED_LEVEL_REVIEW"] = [
    sphc_level_notes.get(sensor, "") if grade == "SPHC" else ""
    for grade, sensor in zip(lot_sensor_summary["GRADE"], lot_sensor_summary["SENSOR"])
]

phase_notes = {
    "SGCC": "240001 단일 LOT. 반복 파형의 독립 정상 기준 미확정",
    "DP590": "240058 단일 LOT. 08:00 이전에도 정기수리 준비 과정이 포함됐는지 확인 필요",
    "SPCD": "240117 단일 LOT. 09:00 이후에도 정비 작업 겹침이 남아 있음",
    "SPHC": "240044·061·069·091 비교. 061 재가동을 구분하고 091 TC-Z7/Z8의 지속 수준 차이도 확인해야 함",
    "SPCE": "240037·121·133 비교. 반복 주기와 정비 감지 맥락을 구분해야 함",
    "SPCEN": "240038·114 비교. 114는 20:00 이후 제외했고 그 이전 종료 준비 여부도 확인 필요",
    "SPFH590": "240024·108 비교. 목표·속도 조건이 같은지는 미확정",
}
grade_sensor_rows = []
for grade, group in retained.groupby("GRADE", sort=True):
    lot_numbers = sorted(group["LOT_NO"].unique())
    for sensor in SENSOR_COLUMNS:
        per_lot = lot_sensor_summary.loc[
            lot_sensor_summary["GRADE"].eq(grade) & lot_sensor_summary["SENSOR"].eq(sensor)
        ]
        medians = per_lot["MEDIAN"].dropna()
        role, paired, needed = sensor_review_context(sensor)
        can_compare = len(lot_numbers) > 1 and len(medians) > 1
        grade_sensor_rows.append({
            "GRADE": grade, "SENSOR": sensor, "SENSOR_ROLE": role,
            "GRADE_RELEVANCE": grade_relevance(sensor), "GRADE_RELEVANCE_SOURCE": GRADE_RELEVANCE_SOURCE,
            "LOT_COUNT": len(lot_numbers), "VALID_LOT_COUNT": len(medians),
            "LOT_LIST": " | ".join(map(str, lot_numbers)),
            "RETAINED_ROWS": len(group), **observed_summary(group[sensor]),
            "LOT_MEDIAN_MIN": medians.min(), "LOT_MEDIAN_MAX": medians.max(),
            "LOT_MEDIAN_RANGE": medians.max() - medians.min(),
            "LOT_MEDIAN_STD": medians.std(ddof=1),
            "CROSS_LOT_COMPARISON_AVAILABLE": can_compare,
            "COMPARISON_SCOPE": "동일 강종 LOT 간 비교 가능; 정상 기준 미확정" if can_compare else "동일 강종 LOT 간 비교 근거 부족; LOT 내부 관찰만 가능",
            "PAIRED_SENSOR": paired, "PHASE_CAUTION": phase_notes.get(grade, "운전단계 확인 필요"),
            "SUSTAINED_LEVEL_REVIEW": sphc_level_notes.get(sensor, "") if grade == "SPHC" else "",
            "REQUIRED_DOMAIN_INPUT": needed,
            "NORMAL_RULE_STATUS": "미확정", "ANOMALY_RULE_STATUS": "미확정",
            "REVIEW_STATUS": "미확정", "MODEL_INPUT_ALLOWED": False,
            "SUMMARY_ROLE": "관측 요약; 정상 허용치 아님",
            "EVIDENCE_SOURCE": "원본 온도·강종변경 + 사용자 지정 시간 제외; docs/02-data-contract.md",
        })
grade_sensor_review = pd.DataFrame(grade_sensor_rows)
assert len(grade_sensor_review) == retained["GRADE"].nunique() * len(SENSOR_COLUMNS)
assert not grade_sensor_review.duplicated(["GRADE", "SENSOR"]).any()
assert grade_sensor_review["NORMAL_RULE_STATUS"].eq("미확정").all()
assert grade_sensor_review["ANOMALY_RULE_STATUS"].eq("미확정").all()
display(grade_sensor_review[[
    "GRADE", "SENSOR", "LOT_COUNT", "VALID_ROWS", "MISSING_ROWS", "MEDIAN", "Q05", "Q95",
    "LOT_MEDIAN_RANGE", "GRADE_RELEVANCE", "COMPARISON_SCOPE", "REQUIRED_DOMAIN_INPUT",
]])
display(lot_sensor_summary)

## 4. 전체 센서의 지속 후보 확인

강종별 수준 비교는 04, LOT 내부 지속 이탈은 여기서 계산함. 과거 5분 중앙값과 각 LOT·센서의 변동 규모를 기준으로 실제 1초 간격 10초 이상 구간을 찾음. 문턱은 탐색 설정이며 공정 허용치가 아님. OP·TC-Z3 제어반응, 주기 변화, 보조계측을 구분하고 후보를 곧바로 1로 바꾸지 않음.

In [ ]:
from namelesspark_temporal_review import build_temporal_review, TEMPORAL_REVIEW_RULES
sustained_candidates, temporal_coverage = build_temporal_review(retained, lot_context)
assert len(temporal_coverage) == 14 * 22
assert sustained_candidates.DURATION_S.ge(10).all()
display(sustained_candidates.groupby(['LOT_NO', 'SENSOR']).size().rename('지속후보수').reset_index())
# 기존 검토표와의 호환용 빈 컬럼. 과거 CP 공통범위 규칙은 실행하지 않음.
CALCULATE_CP_CANDIDATES = False
row_review['CP_CANDIDATE'] = pd.Series(pd.NA, index=row_review.index, dtype='boolean')
row_review['CP_RUN_ID'] = pd.Series(pd.NA, index=row_review.index, dtype='string')
cp_runs = pd.DataFrame(columns=['CP_RUN_ID', 'LOT_NO', 'START', 'END'])

## 5. 각 구간 앞뒤의 CSV 사건 연결

BM·TBM·CBM 각각의 감지·작업, 부품교체, 정기수리, 조업이벤트를 직전·구간 중·직후로 연결함. 가까운 기록도 시간 차이를 남기며 원인으로 자동 판정하지 않음.
DOWN_MIN은 정지 분량이고 실제 정지 시점은 없음. 정기수리는 계획이며 실제 정지 로그가 아님. 정기수리 기간 중에도 강종변경 기록이 있어 연속 정지로 간주하지 않음.
각 원본 행은 LOT 맥락과 겹치는 센서 후보 ID로 근거표를 찾아볼 수 있음. 후보 ID가 없다는 이유로 0을 채우지 않음.

In [ ]:
from namelesspark_domain_context import build_domain_context

# 지속 후보와 모든 LOT의 운전 맥락을 구분함. LOT 범위 자체는 이상 구간이 아님.
lot_windows = retained.groupby('LOT_NO', as_index=False).agg(START=('MEAS_DT', 'min'), END=('MEAS_DT', 'max'))
lot_windows['GRADE'] = lot_windows.LOT_NO.map(lot_context.set_index('LOT_NO').GRADE)
lot_windows['SENSOR'] = 'ALL_SENSORS_CONTEXT'
lot_windows['KIND'] = 'LOT_CONTEXT_ONLY'
domain_intervals = pd.concat([sustained_candidates, lot_windows], ignore_index=True).sort_values(
    ['LOT_NO', 'START', 'END', 'SENSOR']).reset_index(drop=True)
domain_intervals['INTERVAL_ID'] = [f'DOM-{i:04d}' for i in range(1, len(domain_intervals)+1)]
domain_context = build_domain_context(domain_intervals, sources, maintenance, lot_context)
domain_context['MODEL_INPUT_ALLOWED'] = False
assert set(domain_context.INTERVAL_ID) == set(domain_intervals.INTERVAL_ID)
assert not domain_context.CAUSALITY_CONFIRMED.any()

# 기존 LOT 검토표에서도 같은 정비 근거를 사용함.
lot_evidence = domain_context.loc[
    domain_context.CANDIDATE_KIND.eq('LOT_CONTEXT_ONLY')
    & domain_context.SOURCE_KIND.isin(['maintenance_work', 'maintenance_detection'])
    & domain_context.EVIDENCE_FOUND
].copy()
lot_evidence['CONTEXT_KIND'] = lot_evidence.SOURCE_KIND + ':' + lot_evidence.RELATION
lot_evidence = lot_evidence.sort_values(['LOT_NO', 'GAP_HOURS']).drop_duplicates(['LOT_NO', 'SOURCE_WO_NO'])
maintenance_columns = [
    'WO_NO', 'MNT_TYPE', 'DETC_DT', 'STRT_DT', 'END_DT', 'FAIL_MODE', 'REMARK',
    'DOWN_MIN', 'WORK_MIN', 'DOWN_RECORD_STATUS', 'DOWN_TO_WORK_RATIO',
    'DOWN_EXCEEDS_WORK', 'STOP_INTERVAL_KNOWN', 'PART_CHG', 'HAS_PART_RECORD',
    'PART_RECORD_COUNT', 'PART_CODES', 'PART_NAMES', 'PART_QTY_TOTAL', 'PART_CHANGE_REASONS',
]
lot_maintenance_evidence = lot_evidence[
    ['LOT_NO', 'GRADE', 'CONTEXT_KIND', 'GAP_HOURS'] + ['SOURCE_'+c for c in maintenance_columns]
].rename(columns={'SOURCE_'+c:c for c in maintenance_columns}).sort_values(['LOT_NO', 'DETC_DT'])
lot_maintenance_evidence['EVIDENCE_TYPE'] = '정비·부품교체 사후 맥락'
lot_maintenance_evidence['REVIEW_STATUS'] = '미확정'
lot_maintenance_evidence['MODEL_INPUT_ALLOWED'] = False
assert not lot_maintenance_evidence.duplicated(['LOT_NO', 'WO_NO']).any()

# 하나의 행에서 해당 LOT 맥락과 센서 후보의 근거를 찾아갈 수 있게 연결함.
domain_row_index = retained[['LOT_NO', 'MEAS_DT', 'GRADE']].sort_values(['LOT_NO', 'MEAS_DT']).reset_index(drop=True)
lot_ids = domain_intervals.loc[domain_intervals.KIND.eq('LOT_CONTEXT_ONLY')].set_index('LOT_NO').INTERVAL_ID
domain_row_index['LOT_CONTEXT_ID'] = domain_row_index.LOT_NO.map(lot_ids)
domain_row_index['SENSOR_CANDIDATE_IDS'] = ''
for interval in domain_intervals.loc[~domain_intervals.KIND.eq('LOT_CONTEXT_ONLY')].itertuples(index=False):
    hit = domain_row_index.LOT_NO.eq(interval.LOT_NO) & domain_row_index.MEAS_DT.between(interval.START, interval.END)
    domain_row_index.loc[hit, 'SENSOR_CANDIDATE_IDS'] += interval.INTERVAL_ID + '|'
domain_row_index['SENSOR_CANDIDATE_IDS'] = domain_row_index.SENSOR_CANDIDATE_IDS.str.rstrip('|')
domain_row_index['LABEL'] = pd.Series(pd.NA, index=domain_row_index.index, dtype='Int8')
domain_row_index['REVIEW_STATUS'] = '미판정: 센서·운전조건·CSV 사건 근거 종합 필요'
assert len(domain_row_index) == len(retained)
assert not domain_row_index.duplicated(['LOT_NO', 'MEAS_DT']).any()
assert domain_row_index.LOT_CONTEXT_ID.notna().all()
display(lot_maintenance_evidence)
print('검토 구간:', len(domain_intervals), '/ 원본 행 근거 연결:', len(domain_row_index))

## 6. 라벨은 미확정으로 두고 근거표부터 남김

강종·운전단계와 TC·OP 관계를 확인하기 전에 TC 진동, OP 0~100 왕복, 가까운 정비만으로 1을 만들지 않음.
REASON과 EVIDENCE_TYPE은 검토 근거이며 MODEL_INPUT_ALLOWED=False는 이 근거표를 그대로 모델 입력에 쓰지 않는다는 뜻임.
이 05 검토표는 현장 확정 구간이 없어서 LABEL은 NA, REVIEW_STATUS는 미확정으로 남김. 실험에 쓸 임시 0·1은 07에서 생성하는 processed/cal01_base_v1.csv를 기준으로 보면 됨.
후보가 없거나 값이 안정적이라는 이유로 자동으로 정상 0을 채우지 않음.

In [ ]:
row_review["CP_CANDIDATE_CALCULATED"] = CALCULATE_CP_CANDIDATES
row_review["LABEL"] = pd.Series(pd.NA, index=row_review.index, dtype="Int8")
row_review["REVIEW_STATUS"] = "미확정"
row_review["EVIDENCE_TYPE"] = "판정 대기"
row_review["LABEL_REASON"] = "도메인 기준과 구간 판정 미확정. 정상 0으로 확정한 행이 아님"
row_review.loc[row_review["CP_CANDIDATE"].fillna(False), ["EVIDENCE_TYPE", "LABEL_REASON"]] = [
    "CP 신호 후보", "CP 지속 이탈 후보임. 공정 허용치와 정비 맥락 확인 전 이상 1로 확정하지 않음"
]
row_review.loc[row_review["EXCLUDED"], ["EVIDENCE_TYPE", "LABEL_REASON"]] = [
    "사용자 지정 시간 제외", "사용자가 지정한 시간 범위여서 제외함. 정상·이상 판정과 별개임"
]
assert row_review["LABEL"].isna().all()
assert row_review["REVIEW_STATUS"].eq("미확정").all()
assert row_review.loc[row_review["EXCLUDED"], "CP_CANDIDATE"].isna().all()

review_notes = {
    240001: "TC·OP 반복 변동만으로 이상 확정하지 않음. 원인 확인 필요함",
    240024: "OP 하강·TC 상승·CP 변동 관계를 같이 확인해야 함",
    240037: "OP 반복 변동만으로 이상 확정하지 않음. CP 후보는 따로 검토함",
    240038: "TC·OP 반응과 CP 경계의 작은 초과를 구분해서 확인해야 함",
    240044: "초반·후반 OP 결측 처리와 후속 BM 내용을 따로 확인해야 함",
    240058: "08:00 이후 제외 확정. 앞쪽 CP 저하가 정기수리 준비 과정인지 확인해야 함",
    240061: "02:30 이전 제외 확정. 남은 출력만으로 이상 확정하지 않음",
    240069: "CP 변화와 절연 저하·O링 교체를 같이 보되 가스 누출로 확정하지 않음",
    240091: "같은 SPHC보다 TC-Z7/Z8이 높게 지속됨. 짧은 급변과 별도로 설정·운전조건 및 절연 저하 기록을 검토해야 함",
    240108: "OP 변동만으로 원인 확정하지 않음. 정비 기록과 같이 검토함",
    240114: "20:00 이후 제외 확정. 이전 TC·OP 제어 반응을 검토함",
    240117: "09:00 이전 제외 확정. 정비 후 CP 저하의 작업 맥락 확인이 필요함",
    240121: "급격한 TC 변화가 운전 반응인지 이상인지 기준 확인이 필요함",
    240133: "CBM 감지 사실과 실제 센서 구간을 나눠서 검토해야 함",
}
lot_review = row_review.groupby("LOT_NO", as_index=False).agg(
    TOTAL_ROWS=("MEAS_DT", "size"), EXCLUDED_ROWS=("EXCLUDED", "sum"),
    CP_CANDIDATE_ROWS=("CP_CANDIDATE", lambda s: int(s.sum())),
    CP_NOT_EVALUATED_ROWS=("CP_CANDIDATE", lambda s: int(s.isna().sum())),
    CONFIRMED_LABEL_ROWS=("LABEL", lambda s: int(s.notna().sum())),
)
lot_review["RETAINED_ROWS"] = lot_review["TOTAL_ROWS"] - lot_review["EXCLUDED_ROWS"]
lot_review = lot_context.merge(lot_review, on="LOT_NO", how="left", validate="one_to_one")
related_wo = lot_maintenance_evidence.groupby("LOT_NO")["WO_NO"].agg(lambda s: " | ".join(s))
lot_review["CONTEXT_WO_NOS"] = lot_review["LOT_NO"].map(related_wo).fillna("")
lot_review["CP_CANDIDATE_CALCULATED"] = CALCULATE_CP_CANDIDATES
if not CALCULATE_CP_CANDIDATES:
    lot_review["CP_CANDIDATE_ROWS"] = pd.Series(pd.NA, index=lot_review.index, dtype="Int64")
lot_review["REASON"] = lot_review["LOT_NO"].map(review_notes).fillna("별도 도메인 검토 필요함")
lot_review["EVIDENCE_TYPE"] = "LOT·강종·정비 맥락 검토"
lot_review["REVIEW_STATUS"] = "미확정"
lot_review["MODEL_INPUT_ALLOWED"] = False
assert int(lot_review["CP_CANDIDATE_ROWS"].sum()) == int(row_review["CP_CANDIDATE"].sum())
assert int(lot_review["CONFIRMED_LABEL_ROWS"].sum()) == 0
display(lot_review)

## 7. 검토 결과와 확정 구간 전달

row_review는 원본 행·컬럼을 보존하고, CSV는 원본 센서를 복제하지 않고 LOT_NO·MEAS_DT에 연결할 검토 정보만 저장함.
행별 검토·제외·전체 센서 지속 후보·LOT·정비 맥락 표를 저장함. 확정 판정과 별도로 실험용 0·1 생성은 07에서 진행함.
현장 확인으로 확정된 구간이 생기면 review/v1/05_confirmed_intervals.csv에 GRADE, LOT_NO, START, END, LABEL, REASON, EVIDENCE, REVIEWER를 기록할 수 있음. 현재 07은 사용자가 선택한 첫 실험용 임시 0·1 기준을 적용하며, 확정표와 섞어서 사용하지 않음.
LABEL은 확정한 0 또는 1이며, 현재 그 파일을 임의로 만들거나 검토표의 NA를 0으로 채우지 않음.

In [ ]:
REVIEW_COLUMNS = [
    "LOT_NO", "MEAS_DT", "GRADE", "EXCLUDED", "EXCLUSION_REASON",
    "CP_CANDIDATE_CALCULATED", "CP_CANDIDATE", "CP_RUN_ID", "LABEL", "REVIEW_STATUS", "EVIDENCE_TYPE", "LABEL_REASON",
]
review_export = row_review[REVIEW_COLUMNS].copy()
assert len(review_export) == len(temp)
assert review_export["LABEL"].isna().all()
assert not review_export.duplicated(["LOT_NO", "MEAS_DT"]).any()
pd.testing.assert_frame_equal(row_review[temp.columns], temp)

outputs = {
    "05_row_review.csv": review_export,
    "05_domain_intervals.csv": domain_intervals,
    "05_domain_context.csv": domain_context,
    "05_domain_row_index.csv": domain_row_index,
    "05_exclusion_summary.csv": exclusion_summary,
    "05_all_sensor_sustained_candidates.csv": sustained_candidates,
    "05_temporal_channel_coverage.csv": temporal_coverage,
    "05_lot_review.csv": lot_review,
    "05_lot_maintenance_evidence.csv": lot_maintenance_evidence,
    "05_grade_sensor_review.csv": grade_sensor_review,
    "05_lot_sensor_summary.csv": lot_sensor_summary,
}
output_directories = {
    "05_grade_sensor_review.csv": GRADE_OUT,
    "05_lot_sensor_summary.csv": GRADE_OUT,
    "05_all_sensor_sustained_candidates.csv": SIGNAL_OUT,
    "05_temporal_channel_coverage.csv": SIGNAL_OUT,
}
for filename, frame in outputs.items():
    directory = output_directories.get(filename, OUT)
    frame.to_csv(directory / filename, index=False, encoding="utf-8-sig")

display(pd.DataFrame([
    {"파일": name, "행수": len(frame), "저장 위치": str(output_directories.get(name, OUT))}
    for name, frame in outputs.items()
]))
print("확정 라벨 행수:", int(row_review["LABEL"].notna().sum()))
import json
(SIGNAL_OUT / '05_temporal_review_rules.json').write_text(json.dumps(TEMPORAL_REVIEW_RULES, ensure_ascii=False, indent=2), encoding='utf-8')